In [1]:
import pandas as pd
import numpy as np
import os

print("Starting Phase-Wise Player Statistics Engineering...")

# 1. Load Data
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# 2. Define the Phases
# Assuming 'over' column is 0-indexed (0 to 19)
def get_phase(over):
    if over <= 5:
        return 'Powerplay'
    elif over <= 14:
        return 'Middle'
    else:
        return 'Death'

deliveries['phase'] = deliveries['over'].apply(get_phase)

# 3. Calculate Batter Stats per Phase
# We need runs scored and balls faced (excluding wides)
batter_data = deliveries[deliveries['extras_type'] != 'wides']

batter_phase = batter_data.groupby(['batter', 'phase']).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count')
).reset_index()

# Calculate Phase Strike Rate (Runs / Balls * 100)
batter_phase['strike_rate'] = ((batter_phase['runs_scored'] / batter_phase['balls_faced']) * 100).round(2)

# Pivot so each phase has its own column (cleaner for ML and UI)
batter_pivot = batter_phase.pivot(index='batter', columns='phase', values=['runs_scored', 'strike_rate']).fillna(0)
batter_pivot.columns = [f'{col[1].lower()}_{col[0]}' for col in batter_pivot.columns]
batter_pivot = batter_pivot.reset_index()

# 4. Calculate Bowler Stats per Phase
# Economy Rate = (Runs Conceded / Overs Bowled)
# Exclude legbyes and byes from bowler's total runs conceded
bowler_data = deliveries[~deliveries['extras_type'].isin(['byes', 'legbyes'])]

bowler_phase = bowler_data.groupby(['bowler', 'phase']).agg(
    runs_conceded=('total_runs', 'sum'),
    legal_balls=('ball', 'count')
).reset_index()

# Convert legal balls to overs (e.g., 8 balls = 1.33 overs)
bowler_phase['overs_bowled'] = bowler_phase['legal_balls'] / 6
bowler_phase['economy_rate'] = (bowler_phase['runs_conceded'] / bowler_phase['overs_bowled']).round(2)

# Pivot for bowlers
bowler_pivot = bowler_phase.pivot(index='bowler', columns='phase', values=['economy_rate', 'overs_bowled']).fillna(0)
bowler_pivot.columns = [f'{col[1].lower()}_{col[0]}' for col in bowler_pivot.columns]
bowler_pivot = bowler_pivot.reset_index()

# 5. Save the Intelligence Data
os.makedirs('../data/processed', exist_ok=True)
batter_pivot.to_csv('../data/processed/batter_phase_stats.csv', index=False)
bowler_pivot.to_csv('../data/processed/bowler_phase_stats.csv', index=False)

print("✅ Phase-wise stats calculated successfully!")
print("\nSample Batter Phase Stats (Death Overs Specialists):")
display(batter_pivot.sort_values(by='death_strike_rate', ascending=False).head(10))

Starting Phase-Wise Player Statistics Engineering...
✅ Phase-wise stats calculated successfully!

Sample Batter Phase Stats (Death Overs Specialists):


,batter,death_runs_scored,middle_runs_scored,powerplay_runs_scored,death_strike_rate,middle_strike_rate,powerplay_strike_rate
652,WG Jacks,28.0,129.0,73.0,560.00,167.53,148.98
181,DT Patil,4.0,9.0,0.0,400.00,75.00,0.00
429,PA Reddy,15.0,46.0,103.0,300.00,90.20,99.04
312,L Wood,9.0,0.0,0.0,300.00,0.00,0.00
577,Salman Butt,16.0,73.0,104.0,266.67,128.07,106.12
97,B Stanlake,5.0,0.0,0.0,250.00,0.00,0.00
485,RM Patidar,72.0,557.0,170.0,240.00,166.27,123.19
606,T Stubbs,270.0,125.0,10.0,238.94,112.61,111.11
503,Ramandeep Singh,95.0,73.0,2.0,237.50,123.73,66.67
214,HM Amla,122.0,194.0,261.0,234.62,124.36,131.16
